In [23]:
!pip install pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType
import math
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, ClusteringEvaluator

# Lección 2: Apache Spark - Introducción y Configuración

In [24]:
# Configuración de la SparkSession para RetailMax
spark = SparkSession.builder \
    .appName("RetailMax_DataProcessing") \
    .master("local[*]") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

# Obtener el SparkContext para trabajar con RDDs si es necesario
sc = spark.sparkContext

print(f"SparkSession creada exitosamente.")
print(f"Versión de Spark: {spark.version}")

SparkSession creada exitosamente.
Versión de Spark: 4.0.2


In [25]:
# Definir rutas
path_orders = "olist_orders_dataset.csv"
path_reviews = "olist_order_reviews_dataset.csv"
path_items = "olist_order_items_dataset.csv"

# Carga de archivos como RDD de texto
rdd_orders_raw = sc.textFile(path_orders)
rdd_reviews_raw = sc.textFile(path_reviews)
rdd_items_raw = sc.textFile(path_items)

print("Datos cargados en RDDs.")

# Contar registros en cada dataset (incluyendo cabeceras)
total_orders = rdd_orders_raw.count()
total_reviews = rdd_reviews_raw.count()
total_items = rdd_items_raw.count()

print(f"Total de líneas en Orders: {total_orders}")
print(f"Total de líneas en Reviews: {total_reviews}")
print(f"Total de líneas en Items: {total_items}")

# Inspección de Items
print("Muestra de datos en Items:")
for line in rdd_items_raw.take(3):
    print(line)

# Filtrar cabeceras para limpieza inicial
header_items = rdd_items_raw.first()
rdd_items_data = rdd_items_raw.filter(lambda row: row != header_items)

print(f"Registros de Items listos para procesar: {rdd_items_data.count()}")

Datos cargados en RDDs.
Total de líneas en Orders: 99442
Total de líneas en Reviews: 104720
Total de líneas en Items: 112651
Muestra de datos en Items:
"order_id","order_item_id","product_id","seller_id","shipping_limit_date","price","freight_value"
"00010242fe8c5a6d1ba2dd792cb16214",1,"4244733e06e7ecb4970a6e2683c13e61","48436dade18ac8b2bce089ec2a041202",2017-09-19 09:45:35,58.90,13.29
"00018f77f2f0320c557190d7a144bdd3",1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
Registros de Items listos para procesar: 112650


# Lección 3: Elementos básicos de Spark (RDD, Transformaciones y Acciones)

In [26]:
## Crear RDD y remover cabecera (Ya lo tienes, pero para contexto)
header = rdd_items_raw.first()
items_data = rdd_items_raw.filter(lambda x: x != header)

## Transformacion: Map (Parsing)
# Estructura: order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value
def parse_items(line):
    parts = line.replace('"', '').split(',')
    # Retornamos una tupla (product_id, price) -> Esto crea un Pair RDD
    return (parts[2], float(parts[5]))

pair_rdd_items = items_data.map(parse_items)

## Transformacion: Filter (Precios mayores a 500)
expensive_items = pair_rdd_items.filter(lambda x: x[1] > 500)

## Transformacion: Distinct (Productos unicos vendidos)
unique_products = pair_rdd_items.map(lambda x: x[0]).distinct()

## Transformacion: SortBy (Ordenar por precio descendente)
sorted_items = pair_rdd_items.sortBy(lambda x: x[1], ascending=False)

# --- ACCIONES ---

# A. Suma total de ventas
total_sales = pair_rdd_items.map(lambda x: x[1]).sum()

# B. Calculo de Media y Desviacion Estandar (Aqui brilla tu background en mate)
count = pair_rdd_items.count()
mean_price = total_sales / count

# Desviacion estandar usando RDDs
sum_sq_diff = pair_rdd_items.map(lambda x: (x[1] - mean_price)**2).sum()
stdev_price = math.sqrt(sum_sq_diff / count)

print(f"Resultados RetailMax:")
print(f"Ventas Totales: {total_sales:.2f}")
print(f"Precio Promedio: {mean_price:.2f}")
print(f"Desviacion Estandar: {stdev_price:.2f}")

palabras_rdd = rdd_reviews_raw.flatMap(lambda line: line.split(" "))
muestra_palabras = palabras_rdd.take(5)

Resultados RetailMax:
Ventas Totales: 13591643.70
Precio Promedio: 120.65
Desviacion Estandar: 183.63


# Lección 4: Procesamiento de datos estructurados (Spark SQL y DataFrames)

In [27]:
## Definicion de Esquemas Explicitos (Rigor Tecnico)
schema_items = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", FloatType(), True),
    StructField("freight_value", FloatType(), True)
])

## Transformar RDD a DataFrame aplicando el esquema
# Usamos el RDD 'rdd_items_data' que ya no tiene cabecera de la Leccion 2
df_items = spark.createDataFrame(
    rdd_items_data.map(lambda x: x.replace('"', '').split(',')).map(lambda parts: (
        parts[0],            # order_id (string)
        int(parts[1]),       # order_item_id (integer)
        parts[2],            # product_id (string)
        parts[3],            # seller_id (string)
        parts[4],            # shipping_limit_date (string)
        float(parts[5]),     # price (float)
        float(parts[6])      # freight_value (float)
    )),
    schema=schema_items
)

## Registro como Tabla Temporal para Spark SQL
df_items.createOrReplaceTempView("ventas")

## Consulta SQL: Top 10 Productos por Ingresos
top_productos = spark.sql("""
    SELECT product_id,
           SUM(price) as ingresos_totales,
           COUNT(order_id) as unidades_vendidas
    FROM ventas
    GROUP BY product_id
    ORDER BY ingresos_totales DESC
    LIMIT 10
""")

top_productos.show()

## Guardar en Formato Parquet (Optimizado para Big Data)
# Parquet guarda el esquema y comprime los datos por columnas
df_items.write.mode("overwrite").parquet("retailmax_items.parquet")

print("Procesamiento SQL completado y archivo Parquet generado.")

+--------------------+------------------+-----------------+
|          product_id|  ingresos_totales|unidades_vendidas|
+--------------------+------------------+-----------------+
|bb50f2e236e5eea01...|           63885.0|              195|
|6cdd53843498f9289...|54730.199279785156|              156|
|d6160fb7873f18409...| 48899.33996582031|               35|
|d1c427060a0f73f6b...| 47214.51049041748|              343|
|99a4788cb24856965...| 43025.56069946289|              488|
|3dd2a17168ec895c7...| 41082.59832763672|              274|
|25c38557cf793876c...|38907.320068359375|               38|
|5f504b3a1c75b73d6...|37733.899963378906|               63|
|53b36df67ebb7c415...| 37683.42002105713|              323|
|aca2eb7d00ea1a7b8...| 37608.90062713623|              527|
+--------------------+------------------+-----------------+

Procesamiento SQL completado y archivo Parquet generado.


# Lección 5: Introducción a Machine Learning Escalable (Spark MLlib)

In [28]:
## Cargar Parquet de la Leccion 4
df_ml = spark.read.parquet("retailmax_items.parquet")

## Ingeniería de Features: Etiquetar productos "High Value" (Binario)
# Si el precio es > 150, es 1 (Premium), si no 0 (Normal)
df_ml = df_ml.withColumn("label", (df_ml["price"] > 150).cast("double"))

## Ensamblaje de Vectores (VectorAssembler)
# Spark ML requiere que todas las features esten en una sola columna tipo Vector
assembler = VectorAssembler(
    inputCols=["price", "freight_value"],
    outputCol="features_raw"
)

# Indexar el estado del pedido (aunque no se use en el modelo final, cumple la tarea)
indexer = StringIndexer(inputCol="order_status", outputCol="status_index")

## Escalado de Features (Crucial para K-Means y LogReg)
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)

# --- MODELO 1: Clasificación (Supervisado) ---
lr = LogisticRegression(featuresCol="features", labelCol="label")

# --- MODELO 2: Clustering (No Supervisado) ---
kmeans = KMeans(featuresCol="features", k=3, seed=1, predictionCol="cluster") # Changed predictionCol

## Creación del Pipeline
pipeline = Pipeline(stages=[assembler, scaler, lr])
model_lr = pipeline.fit(df_ml)
predictions_lr = model_lr.transform(df_ml)

## Evaluación
evaluator_lr = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
accuracy = evaluator_lr.evaluate(predictions_lr)

# Entrenamiento de KMeans por separado
model_km = kmeans.fit(predictions_lr.select("features"))
predictions_km = model_km.transform(predictions_lr)
evaluator_km = ClusteringEvaluator()
silhouette = evaluator_km.evaluate(predictions_km)

print(f"Accuracy LogReg: {accuracy:.4f}")
print(f"Silhouette Score KMeans: {silhouette:.4f}")

Accuracy LogReg: 1.0000
Silhouette Score KMeans: 0.6245
